# FIFA World Cup 2026: Full Tournament Predictions
By Karl Estampador :)

Simulates all 104 matches using the Elo-based model from `model.ipynb`.  
The higher-Elo team always advances.  
Third-place bracket allocation follows FIFA's official 495-combination lookup table.

In [1]:
# Load model functions and data
%run model.ipynb

48 teams in 2026-05-27 Elo snapshot
48 teams loaded; 48 FIFA ranks available
Alias map and Elo lookup ready.
  Spain                               Elo=2165
  USA                                 Elo=1721
  IR Iran                             Elo=1760
  Côte d'Ivoire                       Elo=1676
  Bosnia and Herzegovina              Elo=1594
  Iraq                                Elo=1607
  DR Congo                            Elo=1655
  Czechia                             Elo=1726
  Sweden                              Elo=1719
  Turkey                              Elo=1902
predict_winner and predict_score ready.

Match                               Winner                         Score
---------------------------------------------------------------------------
Spain vs Morocco                   Spain                          2-0  (Elo: 2165 vs 1822)
Argentina vs Jordan                    Argentina                      2-0  (Elo: 2113 vs 1690)
USA vs Turkey                    Turkey      

In [2]:
import pandas as pd
from pathlib import Path

DATA = Path('data')

# Load schedule and team registry
matches_df = pd.read_csv(DATA / 'matches.csv')
teams_df   = pd.read_csv(DATA / 'teams.csv')

# Build team-ID -> name lookup
id_to_name: dict[int, str] = dict(zip(teams_df['id'], teams_df['team_name']))
name_to_group: dict[str, str] = dict(zip(teams_df['team_name'], teams_df['group_letter']))

alloc_df = pd.read_csv(DATA / 'third_place_allocation.csv')

print(f'{len(matches_df)} matches loaded | {len(teams_df)} teams | allocation table: {len(alloc_df)} rows')

104 matches loaded | 48 teams | allocation table: 495 rows


---
## 1. Group Stage Simulation

In [3]:
group_matches = matches_df[matches_df['stage_id'] == 1].copy()

gs_results = []   # list of dicts, one per group-stage match

for _, row in group_matches.iterrows():
    home = id_to_name[int(row['home_team_id'])]
    away = id_to_name[int(row['away_team_id'])]
    hg, ag = predict_score(home, away)
    winner, loser, _, _ = predict_winner(home, away)
    gs_results.append({
        'match':      int(row['match_number']),
        'group':      row['match_label'],
        'home':       home,
        'away':       away,
        'home_goals': hg,
        'away_goals': ag,
        'winner':     winner,
    })

gs_df = pd.DataFrame(gs_results)
print(f'{len(gs_df)} group-stage matches simulated')

72 group-stage matches simulated


### Group Stage Results

In [4]:
display_gs = gs_df[['match','group','home','home_goals','away_goals','away','winner']].copy()
display_gs.columns = ['Match','Group','Home','HG','AG','Away','Winner']

pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 30)

display_gs

,Match,Group,Home,HG,AG,Away,Winner
0,1,Group A,Mexico,2,0,South Africa,Mexico
1,2,Group A,South Korea,1,0,Czechia,South Korea
2,3,Group B,Canada,2,0,Bosnia and Herzegovina,Canada
3,4,Group D,USA,0,2,Paraguay,Paraguay
4,5,Group B,Qatar,0,2,Switzerland,Switzerland
5,6,Group C,Brazil,2,0,Morocco,Brazil
6,7,Group C,Haiti,0,2,Scotland,Scotland
7,8,Group D,Australia,0,2,Turkey,Turkey
8,9,Group E,Germany,2,0,Curaçao,Germany
9,10,Group F,Netherlands,2,0,Japan,Netherlands


---
## 2. Group Standings

In [ ]:
# build standings table for all 12 groups from group-stage results
def build_standings(gs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Tiebreakers (when points equal):
      1. Goal difference (all group games)
      2. Goals scored (all group games)
      3. Elo rating (proxy for FIFA rank)
    """
    stats: dict[str, dict] = {}

    for _, r in gs_df.iterrows():
        for team, gf, ga in [
            (r['home'], r['home_goals'], r['away_goals']),
            (r['away'], r['away_goals'], r['home_goals']),
        ]:
            if team not in stats:
                stats[team] = {'gp': 0, 'w': 0, 'l': 0, 'gf': 0, 'ga': 0, 'pts': 0}
            s = stats[team]
            s['gp'] += 1
            s['gf'] += gf
            s['ga'] += ga
            if gf > ga:
                s['w']   += 1
                s['pts'] += 3
            else:
                s['l'] += 1

    rows = []
    for team, s in stats.items():
        rows.append({
            'team':  team,
            'group': name_to_group[team],
            'gp':    s['gp'],
            'w':     s['w'],
            'l':     s['l'],
            'gf':    s['gf'],
            'ga':    s['ga'],
            'gd':    s['gf'] - s['ga'],
            'pts':   s['pts'],
            'elo':   get_elo(team),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(
        ['group', 'pts', 'gd', 'gf', 'elo'],
        ascending=[True, False, False, False, False]
    ).reset_index(drop=True)

    # Add rank within group (1-4)
    df['rank'] = df.groupby('group').cumcount() + 1
    return df


standings = build_standings(gs_df)
print(standings[['group','rank','team','gp','w','l','gf','ga','gd','pts','elo']].to_string(index=False))

group  rank                   team  gp  w  l  gf  ga  gd  pts  elo
    A     1                 Mexico   3  3  0   6   0   6    9 1860
    A     2            South Korea   3  2  1   3   2   1    6 1752
    A     3                Czechia   3  1  2   2   3  -1    3 1726
    A     4           South Africa   3  0  3   0   6  -6    0 1524
    B     1            Switzerland   3  3  0   6   0   6    9 1889
    B     2                 Canada   3  2  1   4   2   2    6 1784
    B     3 Bosnia and Herzegovina   3  1  2   2   4  -2    3 1594
    B     4                  Qatar   3  0  3   0   6  -6    0 1425
    C     1                 Brazil   3  3  0   6   0   6    9 1984
    C     2                Morocco   3  2  1   4   2   2    6 1822
    C     3               Scotland   3  1  2   2   4  -2    3 1767
    C     4                  Haiti   3  0  3   0   6  -6    0 1532
    D     1                 Turkey   3  3  0   6   0   6    9 1902
    D     2               Paraguay   3  2  1   4   2   2    6 

### Group Standings Summary (Qualification Status)

In [ ]:
def qualify_label(rank: int, team: str, qualified_thirds: set) -> str:
    if rank == 1:
        return '✓ 1st'
    elif rank == 2:
        return '✓ 2nd'
    elif rank == 3:
        return '? 3rd'
    else:
        return '✗ Out'

disp = standings[['group','rank','team','pts','gd','gf','elo']].copy()
disp['status'] = disp['rank'].map({1:'✓ 1st', 2:'✓ 2nd', 3:'→ TBD 3rd', 4:'✗ Eliminated'})
disp.columns = ['Group','Rank','Team','Pts','GD','GF','Elo','Status']
disp

,Group,Rank,Team,Pts,GD,GF,Elo,Status
0,A,1,Mexico,9,6,6,1860,✓ 1st
1,A,2,South Korea,6,1,3,1752,✓ 2nd
2,A,3,Czechia,3,-1,2,1726,→ TBD 3rd
3,A,4,South Africa,0,-6,0,1524,✗ Eliminated
4,B,1,Switzerland,9,6,6,1889,✓ 1st
5,B,2,Canada,6,2,4,1784,✓ 2nd
6,B,3,Bosnia and Herzegovina,3,-2,2,1594,→ TBD 3rd
7,B,4,Qatar,0,-6,0,1425,✗ Eliminated
8,C,1,Brazil,9,6,6,1984,✓ 1st
9,C,2,Morocco,6,2,4,1822,✓ 2nd


---
## 3. Third-Place Team Ranking

All 12 third-place finishers are ranked by: points → GD → GF → Elo.  
The top 8 advance to the Round of 32.

In [7]:
thirds = standings[standings['rank'] == 3].sort_values(
    ['pts', 'gd', 'gf', 'elo'],
    ascending=[False, False, False, False]
).reset_index(drop=True)

thirds['third_rank'] = range(1, len(thirds) + 1)
thirds['advances'] = thirds['third_rank'] <= 8

print('All third-place teams ranked (top 8 advance):\n')
display_thirds = thirds[['third_rank','group','team','pts','gd','gf','elo','advances']].copy()
display_thirds.columns = ['Rank','Group','Team','Pts','GD','GF','Elo','Advances']
display_thirds

All third-place teams ranked (top 8 advance):



,Rank,Group,Team,Pts,GD,GF,Elo,Advances
0,1,I,Senegal,3,-1,2,1878,True
1,2,A,Czechia,3,-1,2,1726,True
2,3,D,Australia,3,-2,2,1783,True
3,4,C,Scotland,3,-2,2,1767,True
4,5,J,Algeria,3,-2,2,1743,True
5,6,L,Panama,3,-2,2,1737,True
6,7,K,Uzbekistan,3,-2,2,1727,True
7,8,F,Sweden,3,-2,2,1719,True
8,9,G,Egypt,3,-2,2,1689,False
9,10,E,Côte d'Ivoire,3,-2,2,1676,False


---
## 4. Third-Place Bracket Allocation

The 8 qualifying group letters are sorted and looked up in FIFA's official 495-row allocation table, which assigns each third-place team to a specific Round of 32 slot.

In [ ]:
# The 8 advancing third-place teams and their groups
qualifying_thirds = thirds[thirds['advances']]
qualifying_groups_letters = sorted(qualifying_thirds['group'].tolist())
combo_key = ''.join(qualifying_groups_letters)

print(f'Qualifying third-place groups: {qualifying_groups_letters}')
print(f'Combination key: {combo_key}')

alloc_row = alloc_df[alloc_df['qualifying_groups'] == combo_key]

if alloc_row.empty:
    raise ValueError(f'No allocation row found for combination: {combo_key}')

alloc_row = alloc_row.iloc[0]

# Map: match_number -> which group's 3rd-place team plays there
third_slot: dict[int, str] = {}
for col in ['match_75', 'match_78', 'match_79', 'match_80', 'match_81', 'match_82', 'match_85', 'match_88']:
    match_num = int(col.split('_')[1])
    group_letter = alloc_row[col].replace('3', '')
    third_slot[match_num] = group_letter

# Build lookup: group letter → team name (for qualifying thirds)
group_to_third_team: dict[str, str] = dict(zip(qualifying_thirds['group'], qualifying_thirds['team']))

print('\nThird-place match assignments:')
for match_num, grp in sorted(third_slot.items()):
    team = group_to_third_team.get(grp, f'[Group {grp} 3rd]')
    print(f'  Match {match_num}: 3rd place from Group {grp} → {team}')

Qualifying third-place groups: ['A', 'C', 'D', 'F', 'I', 'J', 'K', 'L']
Combination key: ACDFIJKL

Third-place match assignments:
  Match 75: 3rd place from Group D → Australia
  Match 78: 3rd place from Group F → Sweden
  Match 79: 3rd place from Group C → Scotland
  Match 80: 3rd place from Group K → Uzbekistan
  Match 81: 3rd place from Group A → Czechia
  Match 82: 3rd place from Group I → Senegal
  Match 85: 3rd place from Group J → Algeria
  Match 88: 3rd place from Group L → Panama


---
## 5. Knockout Bracket Simulation

Round of 32 → Round of 16 → Quarterfinals → Semifinals → Bronze Final → Final

In [ ]:
import re

# returns the team that finished in position `rank` in the given group
def get_group_finisher(rank: int, group: str) -> str:
    result = standings[(standings['group'] == group) & (standings['rank'] == rank)]
    if result.empty:
        raise ValueError(f'No rank-{rank} team in group {group}')
    return result.iloc[0]['team']

# resolves a bracket token like '1A' into a concrete team name
def resolve_team(token: str,
                 match_winners: dict[int, str],
                 match_losers: dict[int, str]) -> str:
    # Winner of match N
    m = re.match(r'^W(\d+)$', token)
    if m:
        mn = int(m.group(1))
        return match_winners[mn]

    # Runner-up (loser) of match N
    m = re.match(r'^RU(\d+)$', token)
    if m:
        mn = int(m.group(1))
        return match_losers[mn]

    # Group finisher: '1A', '2F', etc.
    m = re.match(r'^([12])([A-L])$', token)
    if m:
        return get_group_finisher(int(m.group(1)), m.group(2))

    # Third-place slot: '3ABCDF' (match number determines which group's 3rd)
    # The calling code handles this separately via third_slot
    raise ValueError(f'Cannot resolve token: {token}')


def parse_label_tokens(label: str) -> tuple[str, str]:
    """Split a match_label like '1E vs 3ABCDF' into two raw tokens."""
    parts = label.strip().split(' vs ')
    if len(parts) != 2:
        raise ValueError(f'Unexpected label format: {label}')
    return parts[0].strip(), parts[1].strip()


# Run knockout bracket
match_winners: dict[int, str] = {}   # match_number -> winner team name
match_losers:  dict[int, str] = {}   # match_number-> loser team name (for bronze)

ko_matches = matches_df[matches_df['stage_id'] >= 2].sort_values('match_number')

stage_names = {2: 'Round of 32', 3: 'Round of 16', 4: 'Quarterfinals',
               5: 'Semifinals', 6: 'Bronze Final', 7: 'Final'}

ko_results = []

for _, row in ko_matches.iterrows():
    mn    = int(row['match_number'])
    stage = int(row['stage_id'])
    label = row['match_label']

    tok_a, tok_b = parse_label_tokens(label)

    # Resolve third-place tokens (e.g. '3ABCDF') using allocation table
    def resolve_with_third(tok):
        if re.match(r'^3[A-L]+$', tok):
            grp = third_slot[mn]
            return group_to_third_team[grp]
        return resolve_team(tok, match_winners, match_losers)

    team_a = resolve_with_third(tok_a)
    team_b = resolve_with_third(tok_b)

    winner, loser, w_elo, l_elo = predict_winner(team_a, team_b)
    match_winners[mn] = winner
    match_losers[mn]  = loser

    ko_results.append({
        'match':  mn,
        'stage':  stage_names[stage],
        'team_a': team_a,
        'team_b': team_b,
        'winner': winner,
        'loser':  loser,
        'w_elo':  int(w_elo),
        'l_elo':  int(l_elo),
    })

ko_df = pd.DataFrame(ko_results)
print(f'{len(ko_df)} knockout matches resolved')

32 knockout matches resolved

---
## 6. Results Output

### Round of 32

In [10]:
def show_stage(stage_name: str):
    df = ko_df[ko_df['stage'] == stage_name][['match','team_a','team_b','winner','w_elo','l_elo']].copy()
    df.columns = ['Match', 'Team A', 'Team B', 'Winner', 'Winner Elo', 'Loser Elo']
    return df.reset_index(drop=True)

show_stage('Round of 32')

,Match,Team A,Team B,Winner,Winner Elo,Loser Elo
0,73,South Korea,Canada,Canada,1784,1752
1,74,Brazil,Japan,Brazil,1984,1904
2,75,Ecuador,Australia,Ecuador,1933,1783
3,76,Netherlands,Morocco,Netherlands,1961,1822
4,77,Germany,Norway,Germany,1923,1912
5,78,France,Sweden,France,2081,1719
6,79,Mexico,Scotland,Mexico,1860,1767
7,80,England,Uzbekistan,England,2020,1727
8,81,Belgium,Czechia,Belgium,1867,1726
9,82,Turkey,Senegal,Turkey,1902,1878


### Round of 16

In [11]:
show_stage('Round of 16')

,Match,Team A,Team B,Winner,Winner Elo,Loser Elo
0,89,Canada,Ecuador,Ecuador,1933,1784
1,90,Brazil,Germany,Brazil,1984,1923
2,91,Netherlands,France,France,2081,1961
3,92,Mexico,England,England,2020,1860
4,93,Spain,Colombia,Spain,2165,1975
5,94,Belgium,Turkey,Turkey,1902,1867
6,95,Paraguay,Portugal,Portugal,1984,1833
7,96,Switzerland,Argentina,Argentina,2113,1889


### Quarterfinals

In [12]:
show_stage('Quarterfinals')

,Match,Team A,Team B,Winner,Winner Elo,Loser Elo
0,97,Ecuador,Brazil,Brazil,1984,1933
1,98,Spain,Turkey,Spain,2165,1902
2,99,France,England,France,2081,2020
3,100,Portugal,Argentina,Argentina,2113,1984


### Semifinals

In [13]:
show_stage('Semifinals')

,Match,Team A,Team B,Winner,Winner Elo,Loser Elo
0,101,Brazil,Spain,Spain,2165,1984
1,102,France,Argentina,Argentina,2113,2081


### Bronze Final (3rd Place)

In [14]:
show_stage('Bronze Final')

,Match,Team A,Team B,Winner,Winner Elo,Loser Elo
0,103,Brazil,France,France,2081,1984


---
## 🏆 Final

In [15]:
final_row = ko_df[ko_df['stage'] == 'Final'].iloc[0]
champion  = final_row['winner']
runner_up = final_row['loser']

print('=' * 60)
print(f'  FINAL  (Match {final_row["match"]})')
print(f'  {final_row["team_a"]:25s} vs  {final_row["team_b"]}')
print(f'  Winner:     {champion}')
print(f'  Runner-up:  {runner_up}')
print('=' * 60)
print(f'\n  2026 FIFA WORLD CUP CHAMPION: {champion.upper()}')
print(f'  (Elo: {int(final_row["w_elo"])}) ')

  FINAL  (Match 104)
  Spain                     vs  Argentina
  Winner:     Spain
  Runner-up:  Argentina

  2026 FIFA WORLD CUP CHAMPION: SPAIN
  (Elo: 2165) 


---
## Full Knockout Bracket Summary

In [16]:
for stage in ['Round of 32', 'Round of 16', 'Quarterfinals', 'Semifinals', 'Bronze Final', 'Final']:
    subset = ko_df[ko_df['stage'] == stage]
    print(f'\n{stage.upper()}')
    print('-' * 60)
    for _, r in subset.iterrows():
        print(f'  Match {r["match"]:3d}: {r["team_a"]:26s} vs {r["team_b"]:26s}  →  {r["winner"]}')


ROUND OF 32
------------------------------------------------------------
  Match  73: South Korea                vs Canada                      →  Canada
  Match  74: Brazil                     vs Japan                       →  Brazil
  Match  75: Ecuador                    vs Australia                   →  Ecuador
  Match  76: Netherlands                vs Morocco                     →  Netherlands
  Match  77: Germany                    vs Norway                      →  Germany
  Match  78: France                     vs Sweden                      →  France
  Match  79: Mexico                     vs Scotland                    →  Mexico
  Match  80: England                    vs Uzbekistan                  →  England
  Match  81: Belgium                    vs Czechia                     →  Belgium
  Match  82: Turkey                     vs Senegal                     →  Turkey
  Match  83: Spain                      vs Austria                     →  Spain
  Match  84: Colombia      